In [5]:
import requests
import pandas as pd
from datetime import datetime, timedelta
import time

In [6]:
# Configuration
API_KEY = 'VeuRJitvw9YkYrN0335Ug9Up6dDp2d7WARUOD7kV'  # Replace with your actual NASA API key
BASE_URL = "https://api.nasa.gov/neo/rest/v1/feed"
START_DATE_STR = "2025-01-01"
END_DATE_STR = "2025-12-31"

In [7]:

current_start = datetime.strptime(START_DATE_STR, "%Y-%m-%d")
final_end = datetime.strptime(END_DATE_STR, "%Y-%m-%d")

In [8]:
all_asteroids = []


print("Starting extraction...")

while current_start <= final_end:
    current_end = current_start + timedelta(days=7)
    
    if current_end > final_end:
        current_end = final_end

    params = {
        'start_date': current_start.strftime("%Y-%m-%d"),
        'end_date': current_end.strftime("%Y-%m-%d"),
        'api_key': API_KEY
    }

    try:
        response = requests.get(BASE_URL, params=params)
        response.raise_for_status()
        data = response.json()

        for date in data['near_earth_objects']:
            for asteroid in data['near_earth_objects'][date]:
                
                # Get close approach data (first entry)
                close_approach = asteroid['close_approach_data'][0] if asteroid.get('close_approach_data') else {}
                
                # Get orbital data
                orbital_data = asteroid.get('orbital_data', {})
                
                # Get estimated diameter
                est_diam = asteroid.get('estimated_diameter', {})
                
                flat_asteroid = {
                    # Basic identification
                    'close_approach_date': date,
                    'id': asteroid.get('id'),
                    'neo_reference_id': asteroid.get('neo_reference_id'),
                    'name': asteroid.get('name'),
                    'designation': asteroid.get('designation'),
                    'nasa_jpl_url': asteroid.get('nasa_jpl_url'),
                    
                    # Physical characteristics
                    'absolute_magnitude_h': asteroid.get('absolute_magnitude_h'),
                    
                    # Estimated diameter - all units
                    'estimated_diameter_min_km': est_diam.get('kilometers', {}).get('estimated_diameter_min'),
                    'estimated_diameter_max_km': est_diam.get('kilometers', {}).get('estimated_diameter_max'),
                    'estimated_diameter_min_m': est_diam.get('meters', {}).get('estimated_diameter_min'),
                    'estimated_diameter_max_m': est_diam.get('meters', {}).get('estimated_diameter_max'),
                    'estimated_diameter_min_miles': est_diam.get('miles', {}).get('estimated_diameter_min'),
                    'estimated_diameter_max_miles': est_diam.get('miles', {}).get('estimated_diameter_max'),
                    'estimated_diameter_min_feet': est_diam.get('feet', {}).get('estimated_diameter_min'),
                    'estimated_diameter_max_feet': est_diam.get('feet', {}).get('estimated_diameter_max'),
                    
                    # Hazard classification
                    'is_potentially_hazardous_asteroid': asteroid.get('is_potentially_hazardous_asteroid'),
                    'is_sentry_object': asteroid.get('is_sentry_object'),
                    
                    # Close approach data
                    'close_approach_date_full': close_approach.get('close_approach_date_full'),
                    'epoch_date_close_approach': close_approach.get('epoch_date_close_approach'),
                    'orbiting_body': close_approach.get('orbiting_body'),
                    
                    # Relative velocity - all units
                    'relative_velocity_km_per_sec': close_approach.get('relative_velocity', {}).get('kilometers_per_second'),
                    'relative_velocity_km_per_hour': close_approach.get('relative_velocity', {}).get('kilometers_per_hour'),
                    'relative_velocity_miles_per_hour': close_approach.get('relative_velocity', {}).get('miles_per_hour'),
                    
                    # Miss distance - all units
                    'miss_distance_astronomical': close_approach.get('miss_distance', {}).get('astronomical'),
                    'miss_distance_lunar': close_approach.get('miss_distance', {}).get('lunar'),
                    'miss_distance_kilometers': close_approach.get('miss_distance', {}).get('kilometers'),
                    'miss_distance_miles': close_approach.get('miss_distance', {}).get('miles'),
                    
                    # Orbital data
                    'orbit_id': orbital_data.get('orbit_id'),
                    'orbit_determination_date': orbital_data.get('orbit_determination_date'),
                    'first_observation_date': orbital_data.get('first_observation_date'),
                    'last_observation_date': orbital_data.get('last_observation_date'),
                    'data_arc_in_days': orbital_data.get('data_arc_in_days'),
                    'observations_used': orbital_data.get('observations_used'),
                    'orbit_uncertainty': orbital_data.get('orbit_uncertainty'),
                    'minimum_orbit_intersection': orbital_data.get('minimum_orbit_intersection'),
                    'jupiter_tisserand_invariant': orbital_data.get('jupiter_tisserand_invariant'),
                    'epoch_osculation': orbital_data.get('epoch_osculation'),
                    'eccentricity': orbital_data.get('eccentricity'),
                    'semi_major_axis': orbital_data.get('semi_major_axis'),
                    'inclination': orbital_data.get('inclination'),
                    'ascending_node_longitude': orbital_data.get('ascending_node_longitude'),
                    'orbital_period': orbital_data.get('orbital_period'),
                    'perihelion_distance': orbital_data.get('perihelion_distance'),
                    'perihelion_argument': orbital_data.get('perihelion_argument'),
                    'aphelion_distance': orbital_data.get('aphelion_distance'),
                    'perihelion_time': orbital_data.get('perihelion_time'),
                    'mean_anomaly': orbital_data.get('mean_anomaly'),
                    'mean_motion': orbital_data.get('mean_motion'),
                    'equinox': orbital_data.get('equinox'),
                    'orbit_class_type': orbital_data.get('orbit_class', {}).get('orbit_class_type'),
                    'orbit_class_description': orbital_data.get('orbit_class', {}).get('orbit_class_description'),
                    'orbit_class_range': orbital_data.get('orbit_class', {}).get('orbit_class_range'),
                }
                
                all_asteroids.append(flat_asteroid)

        print(f"Fetched: {params['start_date']} to {params['end_date']} - Total asteroids: {len(all_asteroids)}")
        
    except Exception as e:
        print(f"Error on {params['start_date']}: {e}")

    current_start = current_end + timedelta(days=1)
    time.sleep(0.5)

print(f"\nExtraction complete! Total asteroids: {len(all_asteroids)}")


Starting extraction...
Fetched: 2025-01-01 to 2025-01-08 - Total asteroids: 124
Fetched: 2025-01-09 to 2025-01-16 - Total asteroids: 249
Fetched: 2025-01-17 to 2025-01-24 - Total asteroids: 376
Fetched: 2025-01-25 to 2025-02-01 - Total asteroids: 501
Fetched: 2025-02-02 to 2025-02-09 - Total asteroids: 626
Fetched: 2025-02-10 to 2025-02-17 - Total asteroids: 767
Fetched: 2025-02-18 to 2025-02-25 - Total asteroids: 928
Fetched: 2025-02-26 to 2025-03-05 - Total asteroids: 1069
Fetched: 2025-03-06 to 2025-03-13 - Total asteroids: 1205
Fetched: 2025-03-14 to 2025-03-21 - Total asteroids: 1340
Fetched: 2025-03-22 to 2025-03-29 - Total asteroids: 1482
Fetched: 2025-03-30 to 2025-04-06 - Total asteroids: 1618
Fetched: 2025-04-07 to 2025-04-14 - Total asteroids: 1724
Fetched: 2025-04-15 to 2025-04-22 - Total asteroids: 1859
Fetched: 2025-04-23 to 2025-04-30 - Total asteroids: 2013
Fetched: 2025-05-01 to 2025-05-08 - Total asteroids: 2129
Fetched: 2025-05-09 to 2025-05-16 - Total asteroids: 223

In [9]:

# Convert list of dicts to DataFrame and save to CSV
df = pd.DataFrame(all_asteroids)
df.to_csv("asteroids_2025_flat_1.csv", index=False)


In [10]:

print(f"\nDone! Saved {len(df)} asteroids to asteroids_2023_flat.csv")


Done! Saved 6101 asteroids to asteroids_2023_flat.csv


In [11]:
df

,close_approach_date,id,neo_reference_id,name,designation,nasa_jpl_url,absolute_magnitude_h,estimated_diameter_min_km,estimated_diameter_max_km,estimated_diameter_min_m,...,perihelion_distance,perihelion_argument,aphelion_distance,perihelion_time,mean_anomaly,mean_motion,equinox,orbit_class_type,orbit_class_description,orbit_class_range
0,2025-01-07,2226514,2226514,226514 (2003 UX34),None,https://ssd.jpl.nasa.gov/tools/sbdb_lookup.htm...,20.16,0.246919,0.552128,246.919266,...,None,None,None,None,None,None,None,None,None,None
1,2025-01-07,2438017,2438017,438017 (2003 YO3),None,https://ssd.jpl.nasa.gov/tools/sbdb_lookup.htm...,18.54,0.520661,1.164233,520.660914,...,None,None,None,None,None,None,None,None,None,None
2,2025-01-07,2481442,2481442,481442 (2006 WO3),None,https://ssd.jpl.nasa.gov/tools/sbdb_lookup.htm...,21.58,0.128397,0.287104,128.397030,...,None,None,None,None,None,None,None,None,None,None
3,2025-01-07,3485806,3485806,(2010 AL60),None,https://ssd.jpl.nasa.gov/tools/sbdb_lookup.htm...,22.29,0.092588,0.207033,92.588058,...,None,None,None,None,None,None,None,None,None,None
4,2025-01-07,3723888,3723888,(2015 NU2),None,https://ssd.jpl.nasa.gov/tools/sbdb_lookup.htm...,20.91,0.174805,0.390877,174.805453,...,None,None,None,None,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6096,2025-12-31,54567023,54567023,(2025 XL5),None,https://ssd.jpl.nasa.gov/tools/sbdb_lookup.htm...,24.20,0.038420,0.085909,38.419789,...,None,None,None,None,None,None,None,None,None,None
6097,2025-12-31,54570437,54570437,(2025 YS5),None,https://ssd.jpl.nasa.gov/tools/sbdb_lookup.htm...,23.01,0.066459,0.148607,66.459180,...,None,None,None,None,None,None,None,None,None,None
6098,2025-12-31,54574840,54574840,(2026 AD),None,https://ssd.jpl.nasa.gov/tools/sbdb_lookup.htm...,24.00,0.042126,0.094198,42.126461,...,None,None,None,None,None,None,None,None,None,None
6099,2025-12-31,54575245,54575245,(2026 AP1),None,https://ssd.jpl.nasa.gov/tools/sbdb_lookup.htm...,21.14,0.157237,0.351593,157.237082,...,None,None,None,None,None,None,None,None,None,None
